# Прогноз дыхательного изменения анатомии по одной КТ

**Статус на 01.10.2026:** выполнен технический расчёт лёгких и связанного с
ними положения сердца по одной КТ Ника без обучения модели. На трёх случаях
открытого набора Learn2Reg LungCT расчёт сопоставлен с удержанными КТ выдоха и
281 анатомическим ориентиром. При выбранной привязке стадий один случай
улучшился относительно неподвижного контроля, два ухудшились. Это проверка
конкретной настройки, а не оценка точности для Ника или итоговая валидация
всей модели.

## Исходное состояние КТ и проверяемый вопрос

По прямому уточнению автора КТ добровольцев с псевдонимами **Ник, Георг, Адам
и Ярослав** были получены **на глубоком вдохе с задержкой дыхания**. Это
сведение о протоколе съёмки, сообщённое автором; оно не выведено из меток
сердечных фаз или автоматически распознано по DICOM. Для каждой выбранной серии
надо отдельно установить её связь с этим протоколом, поле зрения и положение
тела. Фазы кардиосинхронизированной КТ не следует считать фазами дыхания.

Вопрос исследования: насколько хорошо по одной КТ глубокого вдоха можно
предсказать изменение **формы и положения** пяти взаимно согласованных областей
грудной клетки — лёгких, сердца, костей, жира и остальных мягких тканей — при
изменении дыхательного состояния? Готовая модель MONAI Physio непосредственно
выдаёт смещения поверхности лёгких. Смещение сердца в дыхательном опыте
получается при распространении рассчитанного лёгочного поля на область сердца.
Движение костей, жира и прочих тканей этим не становится отдельно обученным
прогнозом.

Неизвестную конкретную геометрию выдоха нельзя однозначно восстановить из
одного тома вдоха. Выход рассматривается как **условный сценарий**, для которого
должны быть указаны стадия модели, исходное дыхательное состояние, алгоритм
распространения поля и неопределённость.

## Данные, доступные для постановки и контроля

| Источник | Установленное свойство | Роль и ограничение |
|---|---|---|
| КТ четырёх добровольцев | Состояние глубокого вдоха с задержкой подтверждено автором для исходных исследований | Индивидуальная исходная анатомия; отдельного подтверждённого выдоха у этих же людей пока нет |
| Полноохватная КТ Ника | В серии грудной клетки найдены 241 срез и продольный охват 300 мм; сердечная серия 307 имеет 12×256 кадров и охват около 159 мм | Первую использовать как кандидат полного входа для лёгких; сегментация с ограниченной сердечной серии не заменяет сегментацию на полном поле зрения |
| Исторический каталог `OLD` | По [статусу Breath Geometry](../../breath%20geometry/docs/PROJECT_STATUS.md) в большинстве девяти кардиологических исследований есть две задержки дыхания | Кандидаты на внутрисессионную проверку; вдох и выдох каждой серии требуют подтверждения, наружная грудная стенка и часть лёгких обрезаны |
| Скачанные открытые пары Breath Geometry | В локальном реестре описаны 10 пар DIR-Lab COPDgene и 30 пар Learn2Reg LungCT; имеются также 10 случаев DIR-Lab 4DCT | Независимая проверка движения лёгких и ориентиров; наличие пары не означает наличия точных масок сердца, жира и костей |

Сведения реестра открытых данных берутся из
[текущего статуса Breath Geometry](../../breath%20geometry/docs/PROJECT_STATUS.md)
и его [описания наборов](../../breath%20geometry/docs/OPEN_DATASETS.md).
Историческая ветка Breath Geometry не возобновляется и её отрицательные
результаты регистрации не заменяются автоматически этим опытом. Используются
существующие данные и протоколы контроля. Набор NLST2023 не является
размеченной выборкой вдох–выдох; неоднородные поднаборы Lung250M-4B нельзя
объединять без проверки происхождения пар.

## Модели и происхождение каждого движения

1. **Лёгкие.** Подогнать официальную PCA-модель лёгкого MONAI Physio к
   сегментации одного исходного тома. Готовый `PhysicsNeMo-MGN-Lung-Motion`
   принимает коэффициенты формы и числовую стадию дыхательного цикла, выдаёт
   смещение вершин. Используются опубликованные веса без обучения или
   дообучения на наших данных. Сеть обучена относительно собственной опорной
   стадии `g070`; КТ глубокого вдоха нельзя молча объявить этой стадией.
2. **Сердце при дыхании.** Фиксировать сердечную фазу и переносить её исходную
   поверхность дыхательным полем лёгких с явным правилом затухания за границей
   органа. Это прогноз *положения, обусловленного выбранным полем*, а не
   отдельный сетевой прогноз дыхательной кинематики сердца. Проверять смещение
   центра, поверхности и изменения формы отдельно.
3. **Сердечные сокращения.** Для иллюстрации, аналогичной официальному
   [совместному сценарию](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-13-breathe-and-beat-a-static-clinical-ct),
   можно наложить ранее проверенную историческую сердечную сеть как вторую,
   независимую временную ось. На Нике она не дала убедительного выигрыша над
   неподвижной подогнанной модельной формой по существующей метрике. Поэтому
   суммарная анимация не служит доказательством точности ни дыхания, ни
   сокращения сердца.
4. **Кости, жир и оставшиеся мягкие ткани.** В первичном опыте сохранять
   исходные маски и показывать их перенос тем же пространственным полем как
   исследовательскую визуализацию. До количественного использования требуется
   отдельный контроль: кости не должны неправдоподобно деформироваться, области
   не должны пересекаться или оставлять искусственные пустоты. Готовых
   органоспецифических весов для этих трёх классов в выбранном сценарии нет.

Готовый [односнимочный пример лёгкого](https://project-monai.github.io/monai-physio/tutorials.html#tutorial-00-lung-demo)
и совместный пример задают технический путь, но сами по себе не подтверждают
его точность на наших добровольцах. В примере поле лёгкого сглаживается и вне
лёгкого сохраняет главным образом нормальную составляющую с заданным
затуханием. Поэтому результат для сердца и грудной стенки зависит от этих
настроек; их следует сохранять в манифесте каждого запуска.

## Сравнение, критерии и научное решение

**До вычисления** зафиксировать серию, исходные маски, физическую систему
координат, разрешение, происхождение весов, опорную стадию, сетку стадий,
параметры распространения поля и границы допустимых деформаций. Для первого
опыта сердце рассматривается при одной фиксированной стадии сокращения:
тогда изменение между кадрами относится только к расчётному дыхательному полю.
Отдельная анимация двух ритмов создаётся после этого контроля.

На удержанной реальной дыхательной фазе сравнивать прогноз с **неподвижной
исходной геометрией** и, где доступно, с простым геометрическим переносом.
Отдельно оценивать для лёгких и сердца объём, смещение центра и расстояния
поверхностей; для открытых наборов с независимыми ориентирами — ошибку
положения этих ориентиров. Dice маски не измеряет соответствие анатомических
точек. Для костей проверять сохранение формы каждой кости, а для всей
классификации — перекрытия, пустоты, выход за поле зрения и определитель
якобиана на сетке. Не объединять эти ошибки одним числом.

Пока подтверждённой парной КТ вдох–выдох у наших четырёх добровольцев нет,
индивидуальную точность дыхательного прогноза для них установить нельзя.
Открытые пары позволяют проверить перенос на других людях, но не дают
индивидуальную ошибку Ника, Георга, Адама или Ярослава. Передача геометрии в
FEM допустима лишь как явно обозначенный сценарий чувствительности; для
утверждения о правильном дыхательном электрическом ответе дополнительно нужны
сопоставимые свойства тканей, контакты и проверка на измерениях.

## Выполненный расчёт и источники входов

Для Ника использована серия 301 полной грудной КТ, а не укороченная сердечная
серия 307. Исходные DICOM только прочитаны. Переведённый в NIfTI том и
автоматические маски пяти долей лёгких и сердца, полученные TotalSegmentator v2,
сохранены локально вне Git. Маски лежат на одной физической сетке с томом и не
касаются его границ. Они остаются автоматическими и не являются принятыми
ручными контурами. Поверхность готовой лёгочной модели подогнана к объединению
масок с помощью аффинного ICP и шести коэффициентов PCA. Дополнительный этап
деформируемой регистрации ICON не выполнен: требуемые для него отдельные веса
отсутствовали локально, а запрос библиотеки на их загрузку не прошёл.
В официальном примере исходные лёгкие сегментирует NV-Segment; здесь по решению
проекта применён TotalSegmentator v2. Различия их границ и состава меток могут
влиять на подгонку и являются отдельным ограничением переноса модели.

Для лёгких запущена готовая сеть `PhysicsNeMo-MGN-Lung-Motion`; её веса не
обучались на КТ проекта. Поле смещения распространено на сердце по правилу
официального примера: сглаживание `15 мм`, переход к нормальной составляющей
через `3–8 мм` за границей лёгкого, затухание до нуля на расстоянии `15 мм`.
Кардиосокращение в этом опыте фиксировано. В качестве нуля условного сценария
использована стадия модели `0,7`; физическое соответствие глубокому вдоху
Ника **не установлено**.

| Вход/результат | Происхождение и статус |
|---|---|
| КТ Ника | Измеренный одиночный том на глубоком вдохе с задержкой по сообщению автора |
| Маски Ника | Автоматический TotalSegmentator v2 на этом же томе; независимой ручной приёмки нет |
| Подогнанная поверхность | Результат аффинного ICP и PCA; без ICON, без целевой фазы дыхания |
| Дыхательное поле | Опубликованные веса MONAI Physio; условная заданная стадия |
| Сердце на других стадиях | Перенос поверхности, извлечённой из исходной маски, полем лёгких; самостоятельной дыхательной сети сердца нет |
| Три открытые пары `LungCT_0001–0003` | Вдох (`0001`) каждого случая подан на вход; выдох (`0000`) удержан для сравнения; маски лёгких предоставлены набором, сердце сегментировано автоматически |

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from IPython.display import display

def local_report(variable, relative_name):
    root = os.environ.get(variable)
    if not root:
        raise RuntimeError(f"Задайте {variable} как путь к локальной папке опыта")
    path = Path(root) / relative_name
    if not path.is_file():
        raise FileNotFoundError(path)
    return json.loads(path.read_text(encoding="utf-8"))

nix_fit = local_report("MONAI_PHYSIO_MP10_NIX_DIR", "fit_lung/fit_report.json")
nix_scenario = local_report("MONAI_PHYSIO_MP10_NIX_DIR", "scenarios/scenario_report.json")
open_fit = local_report("MONAI_PHYSIO_MP10_L2R_DIR", "fit_lung_inhale/fit_report.json")
open_scenario = local_report("MONAI_PHYSIO_MP10_L2R_DIR", "scenarios/scenario_report.json")
open_evaluation = local_report("MONAI_PHYSIO_MP10_L2R_DIR", "evaluation.json")
open_cases = [("0001", open_fit, open_scenario, open_evaluation)]
for case in ("0002", "0003"):
    fit = local_report("MONAI_PHYSIO_MP10_L2R_BATCH_DIR", f"fit_{case}/fit_report.json")
    scenario = local_report("MONAI_PHYSIO_MP10_L2R_BATCH_DIR", f"scenarios_{case}/scenario_report.json")
    evaluation = local_report("MONAI_PHYSIO_MP10_L2R_BATCH_DIR", f"evaluation_{case}.json")
    assert fit["subject_alias"] == scenario["subject_alias"] == evaluation["pair_alias"] == f"l2r_{case}"
    open_cases.append((case, fit, scenario, evaluation))
assert nix_fit["subject_alias"] == nix_scenario["subject_alias"] == "nix"
assert open_fit["subject_alias"] == open_scenario["subject_alias"] == "l2r_0001"
print("Прочитаны локальные отчёты; изображения и маски не загружались")

## Результат для КТ Ника

Среднее расстояние до ближайшей вершины в обоих направлениях между
подогнанной и целевой лёгочными поверхностями при выборе каждой восьмой
вершины равно `8,15 мм`.
Это внутренний контроль подгонки, зависящий от дискретизации; он не измеряет
дыхательное движение. Поверхность модели имеет `18` открытых рёбер, поэтому
её `VTK`-объём нельзя использовать как физиологический объём лёгких.

| Стадия модели относительно 0,7 | Среднее смещение вершин лёгких, мм | Среднее смещение поверхности сердца, мм | Смещение центра поверхности сердца, мм |
|---|---:|---:|---:|
| 0,0 | 1,66 | 0,64 | 0,62 |
| 0,5 | 0,68 | 0,25 | 0,25 |
| 0,7 | 0 | 0 | 0 |
| 0,9 | 0,80 | 0,29 | 0,28 |

Все числа в таблице — **результаты модели**, а не измеренное дыхание Ника.
Поскольку подтверждённого тома выдоха Ника нет, ошибка прогноза для него
неизвестна. Текущая подгонка и неопределённая привязка стадии не допускают
передачу этих поверхностей как анатомически проверенной геометрии в FEM.

In [ ]:
rows = []
for item in nix_scenario["stages"]:
    rows.append({
        "Стадия": item["stage"],
        "Лёгкие: среднее смещение, мм": item["lung"]["vertex_displacement_mean_mm"],
        "Сердце: среднее смещение, мм": item["heart"]["vertex_displacement_mean_mm"],
        "Сердце: смещение центра, мм": item["heart"]["centroid_shift_norm_mm"],
    })
display(pd.DataFrame(rows).round(3))
print("Внутренняя ошибка подгонки, мм:", round(nix_fit["nearest_vertex_mean_mm_stride8"], 3))
print("Это расчётный сценарий; парного выдоха Ника для проверки нет")

## Проверка на трёх открытых парах вдох–выдох

В локально сохранённом архиве Learn2Reg LungCT у случаев `LungCT_0001–0003`
есть парные анатомические ориентиры: `81`, `100` и `100` соответственно.
Метаданные архива определяют том `0001` каждого случая как вдох, `0000` —
как выдох. Оба тома каждой пары лежат на общей физической сетке. Вдох
использован для автоматической сегментации и подгонки модели; выдох и его
ориентиры удержаны для проверки. Предоставленные маски лёгких автоматические;
для сердца на обоих томах использован TotalSegmentator v2.

Во всех трёх случаях подгонка проведена на сетке `1,75 × 2,50 × 1,75 мм`.
Каждая выходная лёгочная поверхность имеет `18` открытых рёбер: её объём
не оценивается. Объёмы в таблице ниже рассчитаны только по предоставленным
маскам КТ. Все смещения модели относятся к **условной** привязке
`вдох = 0,7`, `выдох = 0,0`.

| Случай | Ориентиры | Объём лёгких: вдох → выдох, л | Среднее смещение вершин лёгких, мм | Средняя ошибка ориентиров: неподвижно → модель, мм | Центр сердца: по двум маскам → модель, мм |
|---|---:|---:|---:|---:|---:|
| 0001 | 81 | 5,80 → 3,21 | 1,74 | 20,18 → 20,42 | 28,94 → 0,44 |
| 0002 | 100 | 5,88 → 4,93 | 2,31 | 15,60 → 14,98 | 18,70 → 0,94 |
| 0003 | 100 | 4,64 → 3,61 | 4,94 | 8,14 → 11,18 | 16,35 → 1,48 |

Средняя ошибка, взвешенная по числу ориентиров, выросла с `14,27` до
`15,20 мм` (`+0,93 мм`). Это описательная сводка трёх случаев, а не оценка
популяционной точности. Подгонка исходной лёгочной поверхности дала внутренние
расстояния `7,14`, `4,73` и `5,79 мм` соответственно: часть ошибки присутствует
ещё до применения сети движения. В случае 0002 модель уменьшила ошибку
ориентиров на `0,62 мм`; в случаях 0001 и 0003 увеличила на `0,24` и
`3,04 мм`. Стабильного улучшения относительно неподвижной геометрии нет.

Разница центров автоматических масок сердца между КТ во всех трёх случаях
значительно больше смещения, переданного лёгочным полем. Этот контроль
ограничен точностью масок и межсканным положением пациента; он не доказывает
истинную траекторию сердца. Ошибка ориентиров тоже включает возможное общее
смещение пациента между двумя томами: здесь проверяется абсолютное положение
анатомии в координатах пары, без предварительного совмещения выдоха с вдохом.
Не установлено также, какие стадии сети соответствуют глубокому вдоху и
выдоху этих пар. Выбор наиболее выгодной стадии по удержанному выдоху
был бы подгонкой по результату, а не прогнозом из одной КТ. Сводка с точными
числами и хэшами локальных отчётов сохранена в
[`mp10_open_three_pairs_2026-10-01.json`](mp10_open_three_pairs_2026-10-01.json).

**Решение для проверенной конфигурации:** она не допускается как источник
анатомически проверенной геометрии глубокого выдоха или вход для основной
FEM-постановки. Это решение относится к использованным маскам TotalSegmentator,
подгонке без ICON, выбранным стадиям и распространению лёгочного поля.
Другие настройки и малые дыхательные колебания требуют отдельной проверки.

In [ ]:
rows = []
for case, fit, scenario, evaluation in open_cases:
    stage = {float(item["stage"]): item for item in scenario["stages"]}
    control = stage[0.7]["landmarks"]
    candidate = stage[0.0]["landmarks"]
    assert evaluation["modeled_lung_volume_status"] == "not_evaluated_non_watertight_mesh"
    rows.append({
        "Случай": case,
        "Ориентиры": candidate["count"],
        "Контроль: средняя ошибка, мм": control["static_tre_mean_mm"],
        "Модель: средняя ошибка, мм": candidate["predicted_tre_mean_mm"],
        "Изменение ошибки, мм": candidate["predicted_tre_mean_mm"] - control["static_tre_mean_mm"],
        "Сердце: маски, мм": evaluation["observed_heart_centroid_shift_norm_mm"],
        "Сердце: модель, мм": evaluation["modeled_heart_centroid_shift_norm_mm"],
    })
table = pd.DataFrame(rows)
display(table.round(2))
total = table["Ориентиры"].sum()
weighted_static = (table["Ориентиры"] * table["Контроль: средняя ошибка, мм"]).sum() / total
weighted_model = (table["Ориентиры"] * table["Модель: средняя ошибка, мм"]).sum() / total
print(f"Всего {total} ориентир; взвешенная средняя ошибка: {weighted_static:.2f} → {weighted_model:.2f} мм")
print("Объём открытых лёгочных поверхностей модели не интерпретируется")

## Интерактивная трёхмерная визуализация

Следующая ячейка ноутбука показывает сохранённые поверхности лёгких и сердца
Ника в четырёх рассчитанных стадиях. Модель можно вращать и увеличивать;
ползунок выбирает стадию, кнопка запускает последовательность кадров. Оси
показывают координаты LPS в миллиметрах. Ради скорости отображения сетки
упрощаются одинаковым способом во всех кадрах; численные метрики выше
рассчитаны по исходным поверхностям.

Эта последовательность **не является видеозаписью дыхания Ника**: кадры
соответствуют стадиям модели, а интервалы воспроизведения не имеют измеренной
длительности. Для сердца показан перенос лёгочным полем. Кости, жир и остальные
мягкие ткани здесь не показаны, поскольку их дыхательные поверхности ещё не
были рассчитаны и проверены. Ячейка сохраняет самостоятельный интерактивный
HTML только в локальном каталоге опыта. Он содержит индивидуальную геометрию
и поэтому не включается в Git или общий HTML-экспорт ноутбука.

In [ ]:
import os
from pathlib import Path
from IPython.display import HTML, display
from mp_visualize_motion import motion_figure

scenario_dir = Path(os.environ["MONAI_PHYSIO_MP10_NIX_DIR"]) / "scenarios"
stages = ["0.00", "0.50", "0.70", "0.90"]
frames = [
    (stage, {
        "Лёгкие": scenario_dir / f"lung_stage_{stage}.vtp",
        "Сердце": scenario_dir / f"heart_stage_{stage}.vtp",
    })
    for stage in stages
]
figure = motion_figure(
    frames,
    title="Ник: условное дыхательное движение лёгких и сердца",
    subtitle="Дискретные стадии модели; парного выдоха нет. Сердце перенесено полем лёгких.",
    colors={"Лёгкие": "#5caac3", "Сердце": "#d75864"},
    opacity={"Лёгкие": 0.35, "Сердце": 0.95},
    initial_index=2,  # 0.70 — опорная стадия входной КТ
)
display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))
# Для локального просмотра без Jupyter: HTML содержит индивидуальную геометрию.
output = Path(os.environ["MONAI_PHYSIO_MP10_NIX_DIR"]) / "nix_breathing_interactive_3d.html"
figure.write_html(str(output), include_plotlyjs=True)
print("Локальный интерактивный HTML:", output)

## Воспроизведение и дальнейшая проверка

Скрипты `mp10_prepare_ct_series.py`, `mp10_fit_lung_to_ct.py`,
`mp10_lung_heart_scenarios.py`, `mp10_evaluate_open_pair.py`,
`mp10_aggregate_open_pairs.py`, `mp10_lung_model_smoke.py`,
`mp10_render_3d.py` и `mp10_build_notebook.py` выполняют подготовку одиночной
серии, подгонку, условный перенос, оценку по удержанным парам, сводку
результатов, локальную
трёхмерную визуализацию и формирование ноутбука. Локальные отчёты JSON содержат
хэши входов, сетку, параметры поля и отдельные метрики; пути к медицинским
данным и сами изображения не включаются в Git. Ноутбук `21.22` связывает
протокол с таблицами результатов и кодом их чтения. Для выполнения его
аналитических ячеек нужно задать три локальные переменные окружения:
`MONAI_PHYSIO_MP10_NIX_DIR` указывает на папку с `fit_lung/` и `scenarios/`,
`MONAI_PHYSIO_MP10_L2R_DIR` — на папку первого открытого случая с
`fit_lung_inhale/`, `scenarios/` и `evaluation.json`, а
`MONAI_PHYSIO_MP10_L2R_BATCH_DIR` — на папку отчётов случаев 0002 и 0003.
Адреса исходных КТ, масок и весов передаются скриптам аргументами командной
строки и не записаны в ноутбуке.

Следующий проверочный шаг — подтвердить фактические состояния дыхания в
историческом `OLD` и расширить независимую проверку на случаи DIR-Lab; в
использованном архиве Learn2Reg ручные ориентиры имеются у трёх разобранных
обучающих случаев. Перед изменением модели следует отдельно устранить
ошибку подгонки поверхности, определить соответствие стадий дыханию и проверить
топологию выходной геометрии. Только если эти проверки дадут приемлемое
улучшение относительно неподвижного контроля, рассчитывать сценарии для
Георга, Адама и Ярослава и оценивать пригодность для FEM.

## Технический статус модели на дату записи

Официальный архив готовых лёгочных весов выпуска `2026.07.1` размером
58 923 868 байт локально получен во временное хранилище. Его SHA-256 —
`365d575e20e2fb03cd4940670816aaabc51c837779f61e48ff01977682283cdc`;
архив ZIP прошёл проверку целостности. Внутри есть checkpoint, PCA-модель и
опорная поверхность. Контрольный вывод сети на средней модельной форме с
нулевыми коэффициентами PCA успешно выполнен для стадий `0,0`, `0,5`, `0,7`
и `0,9`: средние модули смещения составили соответственно `2,874`, `1,778`,
`0,297` и `3,144 мм`. Этот тест подтверждает работоспособность загруженных
весов и близость `0,7` к их опорной стадии; он **не использует КТ добровольца**
и не измеряет прогностическую точность. Ранее используемые сердечные веса
находятся отдельно в исключённом каталоге `runs/`. Производные КТ, маски и
поверхности сохранены во временной локальной рабочей области; исходные
медицинские данные не перемещались и в Git не добавлялись.